In [3]:
import PyPDF2
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords

# Download stopword biar makin pro
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\syafiq\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import accelerate
print(accelerate.__version__)

1.12.0


In [5]:
def extract_text_from_pdf(file_path):
    try:
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = ""
            for page in reader.pages:
                text += page.extract_text()
            return text
    except Exception as e:
        return f"Error: {e}"

def clean_text_pro(text):
    # [cite_start]Lowercase & hapus karakter selain huruf [cite: 3, 5]
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # [cite_start]Buang Stopwords (kata umum) biar fokus ke skill [cite: 98]
    stop_words = set(stopwords.words('english'))
    words = text.split()
    meaningful_words = [w for w in words if w not in stop_words]
    
    return " ".join(meaningful_words)

In [6]:
# Masukkan Job Description kamu (Versi Inggris biar akurat Bes!)
job_description = """
Looking for a Senior Engineering Manager or Lighting Designer with expertise in 
Adobe InDesign, SketchUp, and Rhino. Must have skills in SQL, C++, 
and project management. Experience in 3D rendering and digital drafting is required.
"""

# Alamat folder kamu (sesuaikan lagi ya!)
folder_path = "../data/resume dataset/data/DESIGNER/" 
loker_bersih = clean_text_pro(job_description)
vectorizer = TfidfVectorizer()
results = []

# Proses semua file
for file_name in os.listdir(folder_path):
    if file_name.endswith(".pdf"):
        path_lengkap = os.path.join(folder_path, file_name)
        raw_text = extract_text_from_pdf(path_lengkap)
        cv_bersih = clean_text_pro(raw_text)
        
        # [cite_start]Hitung Similarity [cite: 31, 38]
        docs = [cv_bersih, loker_bersih]
        tfidf_matrix = vectorizer.fit_transform(docs)
        score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        
        results.append({"Nama File": file_name, "Skor (%)": round(score * 100, 2)})

# Sorting hasil
df_ranking = pd.DataFrame(results).sort_values(by="Skor (%)", ascending=False)
print(df_ranking.head(10))

       Nama File  Skor (%)
64  28663949.pdf     16.22
65  29147100.pdf     13.15
89  42384185.pdf     11.37
58  26790545.pdf     10.95
28  18795567.pdf     10.88
96  67582956.pdf     10.65
69  30968749.pdf     10.58
54  26496059.pdf     10.19
79  37058472.pdf     10.06
30  18979238.pdf      9.74


In [7]:
%pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions
import os

# 1. Inisialisasi Chroma (Lokal di folder 'db_smartpath')
chroma_client = chromadb.PersistentClient(path="./db_smartpath")

# 2. Pakai embedding model yang ringan tapi pinter (jalan di CPU/GPU kamu)
default_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 3. Bikin 'Collection' (ibaratnya tabel di database)
# Kalau sudah ada, dia bakal ngambil yang lama
collection = chroma_client.get_or_create_collection(name="resumes", embedding_function=default_ef)

# 4. Fungsi buat masukin semua file ke database
def ingest_resumes(folder_path):
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".pdf"):
            path_lengkap = os.path.join(folder_path, file_name)
            
            # Ambil teks & bersihkan (pake fungsi kita yang kemarin)
            raw_text = extract_text_from_pdf(path_lengkap)
            cleaned_text = clean_text_pro(raw_text)
            
            # Masukkan ke Chroma
            # ID harus unik, kita pake nama filenya aja
            collection.add(
                documents=[cleaned_text],
                metadatas=[{"filename": file_name}],
                ids=[file_name]
            )
    print(f"Berhasil masukin {collection.count()} CV ke database, Bes!")

# Jalankan! (Pastikan path foldernya bener ya)
ingest_resumes("../data/resume dataset/data/DESIGNER/")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 467.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Berhasil masukin 107 CV ke database, Bes!


In [8]:
def cari_kandidat(lowongan_teks, jumlah_hasil=5):
    lowongan_bersih = clean_text_pro(lowongan_teks)
    # tanya pada database cuyy
    hasil = collection.query(
        query_texts=[lowongan_bersih],
        n_results=jumlah_hasil
    )

    print(f"Hasil Pencarian untuk: '{lowongan_teks[:50]}...' \n")
    for i in range(len(hasil['ids'][0])):
        print(f"Ranking {i+1}: {hasil['ids'][0][i]}")
        print(f"Kemiripan: {round((1 - hasil['distances'][0][i]) * 100, 2)}%\n")

# contoh cara pake
# masukin kriteria
lowongan_saya = "Looking for a Lighting Designer who is expert in AutoCAD and 3D Rendering"
cari_kandidat(lowongan_saya)

Hasil Pencarian untuk: 'Looking for a Lighting Designer who is expert in A...' 

Ranking 1: 42384185.pdf
Kemiripan: 40.68%

Ranking 2: 10748989.pdf
Kemiripan: 38.74%

Ranking 3: 16288901.pdf
Kemiripan: 38.51%

Ranking 4: 26622051.pdf
Kemiripan: 36.26%

Ranking 5: 18198627.pdf
Kemiripan: 35.14%



In [9]:
%pip install transformers accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Jalankan ini di cell baru
%pip install -U accelerate bitsandbytes transformers

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.7 MB 2.4 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/10.7 MB 2.2 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/10.7 MB 2.3 MB/s eta 0:00:05
   ------- -------------------------------- 2.1/10.7 MB 2.2 MB/s eta 0:00:04
   -------- ------------------------------- 2.4/10.7 MB 2.1 MB/s eta 0:00:05
   ---------- ----------------------------- 2.9/10.7 MB 2.0 MB/s eta 0:00:04
   ----------- ---------------------------- 3.1/10.7 MB 2.0 MB/s eta 0:00:04
   ------------- -------------------------- 3.7/10.7 MB 1.9 MB/s eta 0:00:04
   -------------- ------------------------- 3.9/10.7 MB 1.9 MB/s eta 0:00:04
   --------------- ------------------------ 4.2/10.7 MB 1.8 MB/s eta 0:00:04
   ---------------- ----------------------- 4.5/10.7 MB 1.8 MB/s eta 0:00:04
   ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
access_token = os.getenv("HF_TOKEN")

if access_token:
    login(token=access_token)
else:
    print("Waduh Bes, tokennya gak ketemu! Cek file .env ya.")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# setting agar pake GPU
model_id = "google/gemma-1.1-2b-it"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

print("--loading euy kalem, sabar, tenang, lagi masukin ke otak si isha ke GPU--")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # auto pake CUDA maksudnya kl ada
)

def analisis_pake_isha(job_desc, ranking_1_file):
    # 1. Ambil teks CV dari ChromaDB
    data_cv = collection.get(ids=[ranking_1_file])
    teks_cv = data_cv['documents'][0]

    job_desc = job_description

# prompting technique
    prompt = f"""
<start_of_turn>user
Nama kamu Isha, seorang ahli penasihat HR.
Analisis kenapa kandidat ini sangat cocok untuk pekerjaan yang tersedia.

JOB DESCRIPTION:
{job_desc}

CANDIDAT RESUME:
{teks_cv[:2000]}

MOHON BERIKAN:
1. 3 keterampilan paling sesuai.
2. Mengapa mereka di peringkat 1.
3. Pertanyaan wawancara potensial untuk mereka.<end_of_turn>
<start_of_turn>model
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300)

    print("\n" + "="*50)
    print("HASIL ANALISIS ASISTEN ISHA")
    print("="*50)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("model")[-1])

# EKSEKUSIIIII!!!!
loker = "Looking for a Lighting Designer expert in AutoCAD and SketchUp"
analisis_pake_isha(loker, "42384185.pdf") # pake hasil ranking cuyk

Waduh Bes, tokennya gak ketemu! Cek file .env ya.
--loading euy kalem, sabar, tenang, lagi masukin ke otak si isha ke GPU--


Loading weights: 100%|██████████| 164/164 [00:11<00:00, 14.38it/s, Materializing param=model.norm.weight]                               



HASIL ANALISIS ASISTEN ISHA

**1. 3 Keterampilan paling sesuai:**

- Proficient dalam Adobe Creative Suite (InDesign, SketchUp, Rhino)
- Memiliki pengetahuan dasar tentang SQL, C++, dan project management
- Menjalankan pengalaman dalam 3D rendering dan digital drafting

**2. Mengapa mereka di peringkat 1:**

- Keterampilan yang sangat penting dalam pekerjaan ini, karena mereka membutuhkan keterampilan untuk mengelola proyek, mengotomatiskan proses, dan menciptakan desain profesional.
- Keterampilan ini juga sangat penting untuk perusahaan yang menggunakan software Adobe.

**3. Pertanyaan wawancara potensial:**

- Apa jenis proyek yang akan Anda lakukan?
- Apa level kebebasan yang Anda miliki dalam mengelola proyek?
- Apa langkah-langkah yang Anda gunakan untuk mengotomatiskan proses?
- Apa pengalaman Anda dalam mengelola tim?
- Apa cara Anda mengukur keberhasilan proyek?
